Notebook 3 — SQL Data Warehouse & Analytics Engineering

The purpose of this notebook is to design and implement an enterprise-grade retail data warehouse using SQL and the processed Global Superstore dataset.

This notebook focuses on:

* dimensional modeling
* fact and dimension table design
* star schema architecture
* KPI engineering
* SQL analytics
* business intelligence preparation

The resulting warehouse will support:

* Power BI dashboards
* executive reporting
* customer analytics
* forecasting systems
* machine learning workflows


In [1]:
import pandas as pd
import numpy as np

from sqlalchemy import create_engine
import sqlite3

import warnings
warnings.filterwarnings("ignore")

In [2]:
orders=pd.read_csv("C:/Users/jsero/OneDrive/Desktop/Analyst Portfolio/Global Sales Enterprise/processed_orders.csv", )
orders.head()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,city,state,...,loss_order_flag,total_orders,customer_lifetime_value,avg_order_value,product_total_sales,product_total_profit,regional_total_sales,high_sales_anomaly,extreme_discount_flag,negative_profit_flag
0,26341,IN-2013-77878,2013-02-05,2013-02-07,Second Class,JR-16210,Justin Ritter,Corporate,Wollongong,New South Wales,...,1,46,12589.95900,273.694761,15455.8125,-981.0675,1.100185e+06,1,0,1
1,25330,IN-2013-71249,2013-10-17,2013-10-18,First Class,CR-12730,Craig Reiter,Consumer,Brisbane,Queensland,...,0,58,19487.96376,335.999375,30041.5482,5455.9482,1.100185e+06,1,0,0
2,13524,ES-2013-1579342,2013-01-28,2013-01-30,First Class,KM-16375,Katherine Murray,Home Office,Berlin,Berlin,...,1,70,16394.46064,234.206581,11087.9550,-270.4350,2.822303e+06,1,0,1
3,47221,SG-2013-4320,2013-11-05,2013-11-06,Same Day,RH-9495,Rick Hansen,Consumer,Dakar,Dakar,...,0,7,3057.21000,436.744286,2832.9600,311.5200,7.837732e+05,1,0,0
4,22732,IN-2013-42360,2013-06-28,2013-07-01,Second Class,JM-15655,Jim Mitchum,Corporate,Sydney,New South Wales,...,0,68,16294.72706,239.628339,13715.3940,3638.2740,1.100185e+06,1,0,0


________________________

Data Warehouse Architecture

________________________________

SQL DATABASE CONNECTION

1. SQLite Database

In [3]:
engine = create_engine("sqlite:///enterprise_retail.db")
connection = engine.connect()

_______________________________

Fact Table & Dimensions Table

In [4]:
fact_sales = orders[[ "order_id", "order_date", "customer_id", "product_id", "region", "sales", "quantity", "discount",  "profit", "shipping_days", "profit_margin", "high_discount_flag", "loss_order_flag", "high_sales_anomaly", "negative_profit_flag"]]
dim_customer = orders[[ "customer_id", "customer_name", "segment", "country",  "city",  "state", "region"]].drop_duplicates()
dim_product = orders[[ "product_id", "product_name", "category",  "sub_category"]].drop_duplicates()
dim_region = orders[[ "region", "country",  "state",  "city"]].drop_duplicates()
dim_date = pd.DataFrame({ "order_date": pd.to_datetime(    orders["order_date"] ).unique()})

In [5]:
dim_date["year"] = (dim_date["order_date"].dt.year)
dim_date["month"] = (dim_date["order_date"].dt.month)
dim_date["month_name"] = (dim_date["order_date"].dt.month_name())
dim_date["quarter"] = (dim_date["order_date"].dt.quarter)
dim_date["weekday"] = (dim_date["order_date"].dt.day_name())

In [6]:
fact_sales.to_sql( "fact_sales", engine, if_exists="replace", index=False)
dim_customer.to_sql("dim_customer",engine,if_exists="replace",index=False)
dim_product.to_sql("dim_product", engine, if_exists="replace", index=False)
dim_region.to_sql( "dim_region", engine, if_exists="replace", index=False)
dim_date.to_sql( "dim_date", engine, if_exists="replace", index=False)


1427

___________________________________________________

SQL KPI QUERIES

1. Total Revenue 

In [7]:
query = """
SELECT ROUND(SUM(sales), 2) AS total_revenue
FROM fact_sales
"""
pd.read_sql(query, connection)

,total_revenue
0,11994883.24


2. Total Profit

In [8]:
query = """
SELECT ROUND(SUM(profit), 2) AS total_profit
FROM fact_sales
"""
pd.read_sql(query, connection)

,total_profit
0,1394005.68


3. Profit Margin

In [9]:
query = """
SELECT ROUND( SUM(profit) / SUM(sales), 4 ) AS profit_margin
FROM fact_sales
"""
pd.read_sql(query, connection)

,profit_margin
0,0.1162


4. Average Order Value 

In [10]:
query = """
SELECT ROUND( SUM(sales) / COUNT(DISTINCT order_id),   2) AS average_order_value
FROM fact_sales
"""
pd.read_sql(query, connection)

,average_order_value
0,503.88


5. Customer Count

In [11]:
query = """
SELECT COUNT(DISTINCT customer_id) AS total_customers
FROM fact_sales
"""
pd.read_sql(query, connection)

,total_customers
0,1589


_________________________________________________________

Advanced SQL Analytics 

1. Top Customers 

In [12]:
query = """

SELECT customer_id, ROUND(SUM(sales), 2) AS total_sales,ROUND(SUM(profit), 2) AS total_profit
FROM fact_sales
GROUP BY customer_id
ORDER BY total_sales DESC
LIMIT 10
"""
pd.read_sql(query, connection)

,customer_id,total_sales,total_profit
0,TA-21385,35494.57,6250.33
1,GT-14710,34388.16,5154.11
2,TC-20980,34170.63,8767.97
3,SM-20320,31117.33,-1086.36
4,SE-20110,29360.39,5811.69
5,PS-19045,29252.32,4426.20
6,BW-11110,29205.02,3277.03
7,RB-19360,29197.63,8523.95
8,ZC-21910,28472.82,452.50
9,HL-15040,28094.59,7545.63


2. Most Profitable Categories

In [13]:
query = """

SELECT p.category, ROUND(SUM(f.sales), 2) AS total_sales, ROUND(SUM(f.profit), 2) AS total_profit
FROM fact_sales f
JOIN dim_product p
ON f.product_id = p.product_id
GROUP BY p.category
ORDER BY total_profit DESC
"""
pd.read_sql(query, connection)

,category,total_sales,total_profit
0,Technology,4891322.25,691139.16
1,Office Supplies,3957011.99,544183.55
2,Furniture,4277484.50,290325.85


3. Regional Performance 

In [14]:
query = """
SELECT region, ROUND(SUM(sales), 2) AS total_sales,ROUND(SUM(profit), 2)AS total_profit, ROUND( SUM(profit) / SUM(sales), 4) AS profit_margin
FROM fact_sales
GROUP BY region
ORDER BY total_sales DESC
"""
pd.read_sql(query, connection)

,region,total_sales,total_profit,profit_margin
0,Central,2681735.06,294784.15,0.1099
1,South,1515874.84,134918.99,0.0890
2,North,1182889.43,186029.79,0.1573
3,Oceania,1046068.66,114969.31,0.1099
4,Southeast Asia,841523.79,14147.56,0.0168
5,North Asia,814838.72,157496.78,0.1933
6,EMEA,753234.00,42283.68,0.0561
7,Africa,743398.89,84168.51,0.1132
8,Central Asia,724330.83,131081.38,0.1810
9,West,684960.83,100756.97,0.1471


4. Montly Sales Trend

In [15]:
query = """
SELECT strftime('%Y-%m', order_date) AS month, ROUND(SUM(sales), 2) AS total_sales
FROM fact_sales
GROUP BY month
ORDER BY month
"""
pd.read_sql(query, connection)

,month,total_sales
0,2011-01,89531.75
1,2011-02,86376.32
2,2011-03,139946.69
3,2011-04,112221.04
4,2011-05,138074.13
5,2011-06,208063.09
6,2011-07,112576.18
7,2011-08,193523.08
8,2011-09,274393.70
9,2011-10,191935.79


_______________________________________________

SQL VIEWS

1. Executive KPI View

In [16]:
from sqlalchemy import text
connection.execute(text("DROP VIEW IF EXISTS executive_kpis"))

query = """
CREATE VIEW executive_kpis AS
SELECT 
    ROUND(SUM(sales), 2) AS total_sales,   
    ROUND(SUM(profit), 2) AS total_profit,   
    ROUND(SUM(profit) / SUM(sales), 4) AS profit_margin,
    COUNT(DISTINCT order_id) AS total_orders,  
    COUNT(DISTINCT customer_id) AS total_customers
FROM fact_sales
"""

connection.execute(text(query))
connection.commit()

2. Customer Summary View 

In [17]:
connection.execute(text("DROP VIEW IF EXISTS customer_summary"))
query = """
CREATE VIEW customer_summary AS

SELECT customer_id,ROUND(SUM(sales), 2)AS customer_sales,
        ROUND(SUM(profit), 2)AS customer_profit,COUNT(order_id)AS total_orders
FROM fact_sales
GROUP BY customer_id
"""
connection.execute(text(query))
connection.commit()

________________________________________________

Business Intelligence 

1. Sales Dashboard Table

In [18]:
sales_dashboard = pd.read_sql("""
SELECT region, ROUND(SUM(sales), 2) AS total_sales, ROUND(SUM(profit), 2) AS total_profit,COUNT(order_id) AS total_orders
FROM fact_sales
GROUP BY region
""", connection)

In [19]:
sales_dashboard.head()

,region,total_sales,total_profit,total_orders
0,Africa,743398.89,84168.51,4337
1,Canada,63534.51,16994.64,355
2,Caribbean,307039.27,32831.41,1605
3,Central,2681735.06,294784.15,10573
4,Central Asia,724330.83,131081.38,1974


In [20]:
sales_dashboard.to_csv("C:/Users/jsero/OneDrive/Desktop/Analyst Portfolio/Global Sales Enterprise/sales_dashboard_table.csv", index=False)

In [21]:
customer_summary = pd.read_sql("SELECT * FROM customer_summary",connection)
customer_summary.to_csv("C:/Users/jsero/OneDrive/Desktop/Analyst Portfolio/Global Sales Enterprise/customer_summary.csv",index=False)


Key Insights

1. Revenue and Profitability Differ Across Regions
Certain regions generate high sales volumes but lower profit margins, indicating operational inefficiencies or pricing challenges.

2. Product Categories Vary Significantly in Profitability
Some product categories contribute strongly to revenue while generating relatively weak profit performance.

3. Enterprise KPIs Can Be Centralized Through SQL Views
The warehouse architecture enables reusable KPI layers suitable for executive reporting and dashboard systems.

4. Star Schema Architecture Improves Scalability
The dimensional warehouse structure provides a scalable foundation for forecasting, BI reporting, and advanced analytics workflows.


Conclusion
This notebook successfully transformed processed retail transaction data into an enterprise-grade SQL analytics warehouse.
Key accomplishments include:
* dimensional modeling
* star schema design
* fact and dimension table creation
* SQL KPI engineering
* reusable analytics views
* reporting layer generation

The warehouse architecture developed in this notebook establishes the foundation for:
* Power BI dashboards
* executive reporting
* customer analytics
* forecasting systems
* machine learning workflows

This notebook demonstrates enterprise analytics engineering principles commonly used in modern business intelligence environments.
